In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
from odc.geo.xr import ODCExtensionDa
from odc.geo.geobox import GeoBox
from odc.geo.geom import Geometry
from affine import Affine
from odc.geo.crs import CRS
import matplotlib.pyplot as plt
import pyproj

## Load Data

### Berman

In [ ]:
mines = pd.read_stata("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/mines/raw/20150774_data/Data/BCRT_baseline.dta")
mines

In [ ]:
iso_1_id_dict = pd.Series(mines.iso_1.unique()).reset_index(name="iso_1").set_index("iso_1").iloc[:,0].to_dict()
mines["iso_1_id"] = mines.iso_1.map(iso_1_id_dict)

In [ ]:
variables = ["iso_1_id", "nb_mines_a", "nb_diamond", "mainmineral"]

In [ ]:
# https://www.openicpsr.org/openicpsr/project/113068/version/V1/view?path=/openicpsr/113068/fcr:versions/V1/20150774_data/Data/BCRT_baseline.dta&type=file
mines_ds = xr.Dataset.from_dataframe(mines.set_index(["latitude", "longitude", "year"])[variables])
mines_ds = mines_ds.isel(latitude=slice(None,None,-1))
mines_ds.rio.write_crs(
    4326,
    inplace=True
)

In [ ]:
facilites = gpd.read_file("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/fineprint_mining/open_database_mine_production/data/facilities.gpkg")

In [ ]:
facilites[["production_start", "production_end", "activity_status"]].isna().mean()

In [ ]:
facilities_usable = facilites.dropna(subset=["production_start"])
facilities_usable = facilities_usable.loc[((~facilities_usable.geometry.is_empty) & (facilities_usable.geometry.is_valid))]

In [ ]:
fig, ax = plt.subplots()
mines_ds.isel(year=-1).nb_mines_a.plot.imshow(ax=ax)
facilities_usable.clip_by_rect(*mines_ds.rio.bounds()).plot(ax=ax)

In [ ]:
from odc.geo.xr import rasterize

In [ ]:
def add_crs(geom):
    return Geometry(geom, crs = 4326)

In [ ]:
geobox = GeoBox(shape=mines_ds.odc.geobox.shape, crs=mines_ds.odc.geobox.crs, affine=mines_ds.rio.transform())
rasterized = [rasterize(add_crs(geom), geobox) for geom in facilities_usable.geometry]

In [ ]:
facilites_ds = xr.concat(
    rasterized,
    dim = "facility"
)

In [ ]:
facilities_ds_years = dict()
for year in range(1997, 2011):
    subset = ((facilities_usable.production_start <= year) & ((facilities_usable.production_end >= year) | (facilities_usable.production_end.isna()))).values.reshape(-1,1,1)
    facilities_ds_years[year] = (facilites_ds * subset).sum(dim="facility")

In [ ]:
comparison_ds = xr.concat(facilities_ds_years.values(), dim = pd.Index(facilities_ds_years.keys(), name="year"))

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

mines_ds.isel(year=-1).nb_mines_a.plot.imshow(ax=ax[0])
comparison_ds.isel(year=-1).plot.imshow(ax=ax[1])

In [ ]:
mines = pd.read_stata("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/test/dataverse_files/mininglocal.dta")

In [ ]:
mines.int_year.sort_values().unique()

In [ ]:
mines

In [ ]:
mines.columns.values

---
## Test Grid Alignment

The table contains coordinates. Let check whether they identify the top-left coordinate of the grid cells, which is standard.

In [ ]:
mines_ds.iso_1_id.isel(year=0).plot.imshow()

In [ ]:
countries = gpd.read_file("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/misc/processed/stage_1/gadm/gadm_levelADM_0_simplified.gpkg")

In [ ]:
fix, ax = plt.subplots()
mines_ds.iso_1_id.sel(latitude = slice(-19, -16.5), longitude = slice(23.5, 26)).isel(year=0).plot.imshow(ax = ax)
countries.clip((23.5, -19, 26, -16.5)).exterior.plot(ax=ax, color=None)

The grid is indeed correctly identified by the top-left corner.

---

## Test Variables

In [ ]:
import xarray as xr
mines_ds = xr.open_zarr("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/berman_mining/processed/stage_2/berman_mining_timeseries_reprojected.zarr")

In [ ]:
fix, ax = plt.subplots(1, 2, figsize=(15,5))

np.log(mines_ds.nb_mines_a + 1).isel(year=-1).plot.imshow(ax=ax[0])
(mines_ds.nb_mines_a > 0).where(~mines_ds.nb_mines_a.isnull()).isel(year=-1).plot.imshow(ax=ax[1])

In [ ]:
fix, ax = plt.subplots(1, 2, figsize=(15,5))
np.log(mines_ds.nb_mines_a.sel(year=2010) - mines_ds.nb_mines_a.sel(year=2000) + 1).plot.imshow(ax=ax[0])
any_mine_diff = ((mines_ds.nb_mines_a.sel(year=2010) - mines_ds.nb_mines_a.sel(year=2000)) > 0)
any_mine_diff.where(~mines_ds.nb_mines_a.sel(year=2010).isnull()).plot.imshow(ax=ax[1])

In [ ]:
ntl_harm = xr.open_zarr("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/ntl_harm/harmonized/processed/stage_2/ntl_harm_timeseries_reprojected.zarr", consolidated=False, chunks="auto").ntl_harm
ntl_harm = ntl_harm.odc.assign_crs(4326)

In [ ]:
geobox = GeoBox(
    shape = mines_ds.odc.geobox.shape,
    crs = 4326,
    affine = mines_ds.rio.transform()
)

In [ ]:
ntl_harm_reproj = ntl_harm.sel(time=slice("2000-12-31", "2010-12-31")).odc.reproject(geobox, resampling = "average").compute()

ntl_harm_reproj = ntl_harm_reproj.rename({"time": "year"}).assign_coords(year=("year", ntl_harm_reproj.time.to_pandas().dt.year))

ntl_harm_reproj["longitude"] = mines_ds.longitude

ntl_harm_reproj = ntl_harm_reproj.rename("ntl_harm")

In [ ]:
merged = xr.merge([mines_ds, ntl_harm_reproj], compat="override", join="inner")

In [ ]:
merged.isel(band=0, year=0).ntl_harm

In [ ]:
xr.corr(
    (merged.isel(band=0, year=-1).ntl_harm - merged.isel(band=0, year=0).ntl_harm), 
    (merged.isel(band=0, year=-1).nb_mines_a - merged.isel(band=0, year=0).nb_mines_a)
    )

---

In [ ]:
import pandas as pd

In [ ]:
test = pd.read_parquet("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/assembled/modis_subset.parquet/ix=6/iy=10/data.parquet")
test

##

# Fineprint Inference

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
from odc.geo.xr import ODCExtensionDa
import matplotlib.pyplot as plt

In [ ]:
from odc.geo.geom import Geometry, box
def add_crs(geom):
    return Geometry(geom, crs=4326)

def make_box(bounds):
    return box(*bounds, crs=4326)

In [ ]:
mining = gpd.read_file("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/test/fineprint_mining/global_mining_polygons_v2/global_mining_polygons_v2.gpkg")

In [ ]:
berman_mining = pd.read_stata("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/mines/raw/20150774_data/Data/BCRT_baseline.dta")
iso_1_id_dict = pd.Series(berman_mining.iso_1.unique()).reset_index(name="iso_1").set_index("iso_1").iloc[:,0].to_dict()
berman_mining["iso_1_id"] = berman_mining.iso_1.map(iso_1_id_dict)
variables = ["iso_1_id", "nb_mines_a", "nb_diamond", "mainmineral"]
berman_mining_ds = xr.Dataset.from_dataframe(berman_mining.set_index(["latitude", "longitude", "year"])[variables])
berman_mining_ds = berman_mining_ds.isel(latitude=slice(None,None,-1))

In [ ]:
from odc.geo.xr import rasterize
from odc.geo.geobox import GeoBox
from odc.geo.geom import Geometry

In [ ]:
geobox = GeoBox(
    shape = berman_mining_ds.odc.geobox.shape,
    crs = 4326,
    affine = berman_mining_ds.rio.transform()
)

In [ ]:
geom_series = mining.loc[mining.sindex.query(geobox.extent.geom)].geometry

geoms_rasterized = [
    rasterize(Geometry(geom, crs=4326), geobox, all_touched=True) for geom in geom_series
]

mines_counts = xr.concat(geoms_rasterized, dim="mine").sum(dim="mine")
mines_counts = xr.DataArray(np.ma.masked_array(mines_counts.values, berman_mining_ds.nb_mines_a.isel(year=-1).isnull().values))

In [ ]:
geom_merged = geom_series.union_all()

mines_area = (rasterize(Geometry(geom_merged, crs=4326), geobox.zoom_to(resolution=.01), all_touched=True) * 1).odc.reproject(geobox, resampling="sum")

In [ ]:
comparison_ds = xr.Dataset()
comparison_ds = comparison_ds.assign(fineprint_count = (["latitude", "longitude"], mines_counts.values))
comparison_ds = comparison_ds.assign(fineprint_area = (["latitude", "longitude"], mines_area.values))
comparison_ds = comparison_ds.assign(berman = (["latitude", "longitude"], berman_mining_ds.nb_mines_a.isel(year=-1).values))

comparison_ds = comparison_ds.assign_coords(
    {
        "latitude": berman_mining_ds.latitude.values,
        "longitude": berman_mining_ds.longitude.values
    }
    )


In [ ]:
xr.corr(comparison_ds.fineprint_count, comparison_ds.berman)

In [ ]:
xr.corr(comparison_ds.fineprint_area, comparison_ds.berman)

In [ ]:
berman_mining_ds = berman_mining_ds.rio.write_crs(4326)

berman_mining_ds_downsampled = berman_mining_ds.odc.assign_crs(4326).nb_mines_a.odc.reproject(geobox.zoom_to(resolution=.01))

In [ ]:
berman_mining_ds_detail = ((rasterize(Geometry(geom_merged, crs=4326), geobox.zoom_to(resolution=.01), all_touched=True) * 1) * berman_mining_ds_downsampled)

In [ ]:
berman_mining_ds_detail.isel(year=0).plot.imshow()

---

### Combining with Night Lights

In [ ]:
ntl_harm = xr.open_zarr("/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/ntl_harm/harmonized/processed/stage_2/ntl_harm_timeseries_reprojected.zarr", consolidated=False).ntl_harm
ntl_harm = ntl_harm.odc.assign_crs(4326)

In [ ]:
test_mine = mining.sort_values("AREA").iloc[[-6]] #
test_mine

Let's start with an example: The spatially largest mine in the world in Peru. We see that there was no activity in 1992, but increased lights in 2000.

In [ ]:
test_subset = ntl_harm.odc.crop(make_box(test_mine.iloc[0].geometry.buffer(1).bounds)).compute()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5))

for ix, year in enumerate([2016, 2018, 2020, 2022]): #[1992, 2000, 2010, 2020]
    test_subset.sel(time=f"{year}-12-31").squeeze().plot.imshow(ax=ax[ix], robust = True, add_colorbar=False)
    test_mine.plot(ax=ax[ix], alpha=.3, color="red")

In [ ]:
list_files = [
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02B03.V50.A1992201.2021300.hdf",
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02B03.V50.A2002201.2021300.hdf",
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02B03.V50.A2012201.2021300.hdf",
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02B03.V50.A2022201.2023107.hdf"
]

list_files = [
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02E01.V50.A2000201.h29v09.2021254.hdf",
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02E01.V50.A2005201.h29v09.2019248.hdf",
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02E01.V50.A2010201.h29v09.2021228.hdf",
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02E01.V50.A2015201.h29v09.2019040.hdf",
    "/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS02E01.V50.A2020201.h29v09.2021245.hdf"
]

albedo = xr.concat(
    [rxr.open_rasterio(file) for file in list_files],
    dim = "year"#{"year": [1992, 2002, 2012, 2022]}
)

modis_crs = "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +a=6371007.181 +b=6371007.181 +units=m +no_defs"
albedo = albedo.odc.assign_crs(modis_crs)

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 5))

for ix in range(5): #[1992, 2000, 2010, 2020]
    albedo.ABD_WSA_VIS.isel(year=ix).odc.crop(make_box(test_mine.iloc[0].geometry.buffer(1).bounds)).squeeze().plot.imshow(ax=ax[ix], robust = True, add_colorbar=False)
    test_mine.to_crs(modis_crs).plot(ax=ax[ix], alpha=.3, color="red")

In [ ]:
albedo.ABD_WSA_VIS.squeeze().odc.crop(add_crs(test_mine.geometry.iloc[0])).mean(dim=["x","y"]).values

In [ ]:
list_files = ["/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS01E01.V60.A2000201.h29v09.2022017.hdf",
"/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS01E01.V60.A2005201.h29v09.2022011.hdf",
"/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS01E01.V60.A2010201.h29v09.2022011.hdf",
"/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS01E01.V60.A2015201.h29v09.2022011.hdf",
"/scicore/home/meiera/schulz0022/projects/growth-and-temperature/data_nobackup/glass/Albedo/test/GLASS01E01.V60.A2020201.h29v09.2022139.hdf"]

leaf_index = xr.concat(
    [rxr.open_rasterio(file) for file in list_files],
    dim = "year"#{"year": [1992, 2002, 2012, 2022]}
)


modis_crs = "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +a=6371007.181 +b=6371007.181 +units=m +no_defs"
leaf_index = leaf_index.odc.assign_crs(modis_crs)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5))

for ix in range(4): #[1992, 2000, 2010, 2020]
    leaf_index.isel(year=ix).odc.crop(make_box(test_mine.iloc[0].geometry.buffer(1).bounds)).squeeze().plot.imshow(ax=ax[ix], robust = True, add_colorbar=False)
    test_mine.to_crs(modis_crs).plot(ax=ax[ix], alpha=.3, color="red")

In [ ]:
leaf_index.squeeze().odc.crop(add_crs(test_mine.geometry.iloc[0])).mean(dim=["x","y"]).values

We can identify this as a strucutral break.

In [ ]:
def detect_mine_opening(nl, *,
                        min_size=6,
                        early_period=None,
                        early_threshold=1.0,
                        min_abs_jump=1.0,
                        min_rel_reduction=0.05):
    """
    Detect a single upward structural break consistent with a mine "opening"
    (jump from near-zero early activity) using only numpy.

    Parameters:
      nl : 1-D array-like of nonnegative values (sum of nightlights per time unit)
      min_size : minimum observations on each side of a candidate break (default 6)
      early_period : integer window at start to assess whether mine was already open.
                     If None, uses min_size.
      early_threshold : if mean(activity[:early_period]) > early_threshold, assume already open
      min_abs_jump : minimum required (right_mean - left_mean) to call an opening
      min_rel_reduction : minimum fraction reduction in SSE by splitting at the break

    Returns:
      (break_index_or_None, details_dict)
        - break_index_or_None: integer k meaning break between nl[:k] and nl[k:], or None
        - details_dict: diagnostics including 'reason' and candidate stats
    """
    import numpy as np

    x = np.asarray(nl, dtype=float).ravel()
    n = x.size
    if n < 2 * max(1, int(min_size)):
        return None, {'reason': 'series_too_short', 'n': int(n), 'min_required': 2 * int(min_size)}

    early_period = int(min_size) if early_period is None else int(early_period)
    early_period = min(early_period, n)
    early_mean = float(np.mean(x[:early_period]))

    if early_mean > float(early_threshold):
        return None, {'reason': 'already_open_early', 'early_period': early_period, 'early_mean': early_mean,
                      'early_threshold': float(early_threshold)}

    # cumulative sums for fast SSE computation
    s = np.cumsum(x)
    ss = np.cumsum(x * x)
    S_total = s[-1]
    SS_total = ss[-1]
    sse_global = SS_total - (S_total * S_total) / n
    if sse_global <= 0:
        # no variation -- cannot detect a meaningful break
        return None, {'reason': 'no_variation', 'sse_global': float(sse_global)}

    # candidate split points k: left length k in [min_size, n-min_size]
    k_min = int(min_size)
    k_max = int(n - min_size)
    ks = np.arange(k_min, k_max + 1)

    S_k = s[ks - 1]
    SS_k = ss[ks - 1]
    n_k = ks
    n_r = n - n_k

    mean_left = S_k / n_k
    S_r = S_total - S_k
    mean_right = S_r / n_r
    jump = mean_right - mean_left  # positive values indicate upward jump

    sse_left = SS_k - (S_k * S_k) / n_k
    SS_r = SS_total - SS_k
    sse_right = SS_r - (S_r * S_r) / n_r
    sse_split = sse_left + sse_right

    reduction = (sse_global - sse_split) / sse_global

    # Candidate mask: left mean near zero (<= early_threshold), upward jump, and sufficient SSE reduction
    good_mask = (mean_left <= float(early_threshold)) & (jump >= float(min_abs_jump)) & (reduction >= float(min_rel_reduction))

    if not np.any(good_mask):
        # return diagnostics including best candidate even if it failed thresholds
        best_idx = int(np.argmax(reduction))
        best_k = int(ks[best_idx])
        return None, {
            'reason': 'no_upward_break_meeting_thresholds',
            'best_k_by_reduction': best_k,
            'best_left_mean': float(mean_left[best_idx]),
            'best_right_mean': float(mean_right[best_idx]),
            'best_jump': float(jump[best_idx]),
            'best_reduction': float(reduction[best_idx]),
            'thresholds': {
                'early_threshold': float(early_threshold),
                'min_abs_jump': float(min_abs_jump),
                'min_rel_reduction': float(min_rel_reduction)
            }
        }

    # choose among good candidates: prefer highest reduction, break ties by earliest k
    good_indices = np.where(good_mask)[0]
    # reductions for good candidates
    good_reductions = reduction[good_mask]
    best_rel_idx_in_good = int(np.argmax(good_reductions))
    chosen_idx = good_indices[best_rel_idx_in_good]
    chosen_k = int(ks[chosen_idx])

    details = {
        'reason': 'opening_detected',
        'best_k': chosen_k,
        'left_mean': float(mean_left[chosen_idx]),
        'right_mean': float(mean_right[chosen_idx]),
        'jump': float(jump[chosen_idx]),
        'reduction': float(reduction[chosen_idx]),
        'sse_global': float(sse_global),
        'sse_split': float(sse_split[chosen_idx]),
        'thresholds': {
            'early_threshold': float(early_threshold),
            'min_abs_jump': float(min_abs_jump),
            'min_rel_reduction': float(min_rel_reduction)
        }
    }
    return chosen_k, details

In [ ]:
test_ntl_harm_array = test_subset.squeeze().odc.crop(add_crs(test_mine.geometry.iloc[0])).mean(dim=["latitude","longitude"]).values
test_ntl_harm_array

In [ ]:
k, info = detect_mine_opening(test_ntl_harm_array, min_size=1, early_threshold=0.1, min_abs_jump=.1)
print(k, info)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5))

for ix, year in enumerate([1992, 1994, 1996, 1998]):
    test_subset.sel(time=f"{year}-12-31").squeeze().plot.imshow(ax=ax[ix], robust = True, add_colorbar=False)
    test_mine.plot(ax=ax[ix], alpha=.3, color="red")

Let's write a function to test this with other mines

In [ ]:
test_mine.iloc[0].geometry.bounds

In [ ]:
def visualize_opening(mine_geom):
    subset = ntl_harm.odc.crop(make_box(mine_geom.buffer(1).bounds)).compute()
    ntl_harm_array = subset.squeeze().odc.crop(add_crs(mine_geom)).mean(dim=["latitude","longitude"]).values
    k, info = detect_mine_opening(ntl_harm_array, min_size=1, early_threshold=0.1, min_abs_jump=.1)
    
    if not k:
        print(info)
    else:
        fig, ax = plt.subplots(1, 5, figsize=(20, 5))

        for ix, ix_data in enumerate([0] + list(range(k-1, k+2)) + [-1]):
            subset.isel(time=ix_data).squeeze().plot.imshow(ax=ax[ix], robust = True, add_colorbar=False)
            gpd.GeoSeries([mine_geom], crs=4326).plot(ax=ax[ix], alpha=.3, color="red")
            
visualize_opening(mining.sort_values("AREA").iloc[-6].geometry)

In [ ]:
mining.sort_values("AREA").iloc[-6].geometry

In [ ]:
[mining.sort_values("AREA").iloc[-6].geometry.centroid]

In [ ]:
(albedo.BSA_VIS.sel(x = slice(0, 30), y = slice(30, 0)) / 4420).squeeze().plot.imshow()